**3. Treinamento — Detecção de Pessoas e EPIs**

**Objetivo**

Nesta etapa será treinado um modelo de detecção de objetos
utilizando YOLO Ultralytics.

O modelo será ajustado utilizando o dataset de detecção
preparado na etapa anterior, contendo cinco classes:

- person
- with-helmet
- with-suit
- without-helmet
- without-suit

O objetivo é identificar e localizar esses objetos por meio
de bounding boxes.

O modelo será treinado utilizando fine-tuning a partir de
pesos pré-treinados e posteriormente avaliado utilizando
métricas de detecção.

**3.1 Verificar GPU**

In [ ]:
#verifica GPU
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU não disponível")

In [ ]:
!pip install -q ultralytics

In [ ]:
import ultralytics

print("Ultralytics:", ultralytics.__version__)

**3.2 Estruturação do dataset**

In [ ]:
import os

dataset_path = "/content/PPE_dataset_detection"
yaml_path = os.path.join(dataset_path, "data.yaml")

print("Dataset existe:", os.path.exists(dataset_path))
print("data.yaml existe:", os.path.exists(yaml_path))

if os.path.exists(dataset_path):
    print("\nConteúdo do dataset:")
    print(os.listdir(dataset_path))

if os.path.exists(yaml_path):
    print("\nConteúdo do data.yaml:")
    with open(yaml_path, "r") as f:
        print(f.read())

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
#localizar e extrair zip
import os
import zipfile

# Procurar o arquivo ZIP enviado
arquivos_zip = [
    f for f in os.listdir("/content")
    if f.lower().endswith(".zip")
]

print("Arquivos ZIP encontrados:")
for arquivo in arquivos_zip:
    print("-", arquivo)

# Usar o primeiro ZIP encontrado
zip_path = os.path.join("/content", arquivos_zip[0])

# Pasta de extração
extract_path = "/content/PPE_dataset"

# Extrair
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("\nExtração concluída.")
print("Pasta:", extract_path)

In [ ]:
#verificar estrutura do data.yaml
import os

dataset_path = "/content/PPE_dataset"

print("Conteúdo principal:")
for item in os.listdir(dataset_path):
    print("-", item)

print("\nPastas de imagens:")
for split in ["train", "valid", "test"]:
    images_path = os.path.join(dataset_path, split, "images")
    labels_path = os.path.join(dataset_path, split, "labels")

    print(f"\n{split}:")
    print("  images existe:", os.path.exists(images_path))
    print("  labels existe:", os.path.exists(labels_path))

print("\nConteúdo do data.yaml:")
yaml_path = os.path.join(dataset_path, "data.yaml")

with open(yaml_path, "r") as f:
    print(f.read())

In [ ]:
#criar dataset limpo e dataset de detecção
import os
import shutil

# --------------------------------------------------
# 1. Caminhos
# --------------------------------------------------

original_path = "/content/PPE_dataset"
clean_path = "/content/PPE_dataset_clean"
detection_path = "/content/PPE_dataset_detection"

# --------------------------------------------------
# 2. Criar cópia limpa do dataset original
# --------------------------------------------------

if os.path.exists(clean_path):
    shutil.rmtree(clean_path)

shutil.copytree(original_path, clean_path)

print("Cópia limpa criada em:")
print(clean_path)

# --------------------------------------------------
# 3. Arquivos com anotações vazias identificados no EDA
# --------------------------------------------------

arquivos_problematicos = {
    "train": [
        "frameHRGateDay-6-mp422000_jpg.rf.3d2f293e76e6684116e1f37927287c4f.txt",
        "frameHRGateDay-6-mp422000_jpg.rf.8b2b90e67dfdb25c41b31618b7f13a15.txt",
        "frameHRGateDay-6-mp422000_jpg.rf.8f6838da4874ddb2b77afe3157fd8f47.txt"
    ],
    "valid": [
        "frameHRGateDay-6-mp421000_jpg.rf.0a848b789a602dfcb85188607dc2d8ea.txt"
    ],
    "test": []
}

# --------------------------------------------------
# 4. Remover arquivos problemáticos e suas imagens
# --------------------------------------------------

for split, arquivos in arquivos_problematicos.items():

    labels_dir = os.path.join(clean_path, split, "labels")
    images_dir = os.path.join(clean_path, split, "images")

    for label_file in arquivos:

        label_path = os.path.join(labels_dir, label_file)

        # Remover anotação
        if os.path.exists(label_path):
            os.remove(label_path)

        # Remover imagem correspondente
        image_base = os.path.splitext(label_file)[0]

        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            image_path = os.path.join(images_dir, image_base + ext)

            if os.path.exists(image_path):
                os.remove(image_path)

print("Arquivos problemáticos removidos.")

# --------------------------------------------------
# 5. Criar estrutura do dataset de detecção
# --------------------------------------------------

if os.path.exists(detection_path):
    shutil.rmtree(detection_path)

for split in ["train", "valid", "test"]:

    os.makedirs(
        os.path.join(detection_path, split, "images"),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(detection_path, split, "labels"),
        exist_ok=True
    )

print("Estrutura do dataset de detecção criada.")

In [ ]:
#Conversão
def polygon_to_bbox(valores):
    """
    Converte uma anotação YOLO de segmentação
    em uma anotação YOLO de detecção.

    Entrada:
        classe x1 y1 x2 y2 x3 y3 ...

    Saída:
        classe centro_x centro_y largura altura
    """

    classe = int(valores[0])

    coordenadas = list(map(float, valores[1:]))

    xs = coordenadas[0::2]
    ys = coordenadas[1::2]

    x_min = min(xs)
    x_max = max(xs)

    y_min = min(ys)
    y_max = max(ys)

    largura = x_max - x_min
    altura = y_max - y_min

    x_centro = (x_min + x_max) / 2
    y_centro = (y_min + y_max) / 2

    return [
        classe,
        x_centro,
        y_centro,
        largura,
        altura
    ]

print("Função polygon_to_bbox criada com sucesso.")

In [ ]:
#converção das anotações
import os
import shutil

# Extensões de imagem aceitas
image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

# Classes válidas
classes_validas = set(range(5))

erros = []

for split in ["train", "valid", "test"]:

    origem_images = os.path.join(clean_path, split, "images")
    origem_labels = os.path.join(clean_path, split, "labels")

    destino_images = os.path.join(detection_path, split, "images")
    destino_labels = os.path.join(detection_path, split, "labels")

    # Copiar imagens
    for arquivo in os.listdir(origem_images):

        if arquivo.lower().endswith(image_extensions):

            origem = os.path.join(origem_images, arquivo)
            destino = os.path.join(destino_images, arquivo)

            shutil.copy2(origem, destino)

    # Converter labels
    for arquivo in os.listdir(origem_labels):

        if not arquivo.endswith(".txt"):
            continue

        origem_label = os.path.join(origem_labels, arquivo)
        destino_label = os.path.join(destino_labels, arquivo)

        novas_linhas = []

        with open(origem_label, "r") as f:
            linhas = f.readlines()

        for numero_linha, linha in enumerate(linhas, start=1):

            valores = linha.strip().split()

            try:

                # Verificar quantidade mínima de dados
                if len(valores) < 7:
                    raise ValueError(
                        "Anotação não possui pontos suficientes"
                    )

                classe = int(valores[0])

                # Verificar classe
                if classe not in classes_validas:
                    raise ValueError(
                        f"Classe inválida: {classe}"
                    )

                # Converter polígono para bounding box
                bbox = polygon_to_bbox(valores)

                novas_linhas.append(
                    " ".join(f"{valor:.6f}" if i > 0 else str(valor)
                             for i, valor in enumerate(bbox))
                )

            except Exception as e:

                erros.append({
                    "split": split,
                    "arquivo": arquivo,
                    "linha": numero_linha,
                    "erro": str(e)
                })

        # Salvar novo label
        with open(destino_label, "w") as f:
            f.write("\n".join(novas_linhas))

print("Conversão concluída.")
print("Quantidade de erros:", len(erros))

if erros:
    print("\nPrimeiros erros:")
    for erro in erros[:10]:
        print(erro)

In [ ]:
#Validação automática
import os

erros_validacao = []

for split in ["train", "valid", "test"]:

    images_dir = os.path.join(detection_path, split, "images")
    labels_dir = os.path.join(detection_path, split, "labels")

    imagens = [
        f for f in os.listdir(images_dir)
        if f.lower().endswith(image_extensions)
    ]

    labels = [
        f for f in os.listdir(labels_dir)
        if f.endswith(".txt")
    ]

    print(f"\n{split}:")
    print("  Imagens:", len(imagens))
    print("  Labels:", len(labels))

    # Verificar se cada imagem possui label
    for imagem in imagens:

        nome_base = os.path.splitext(imagem)[0]
        label = os.path.join(labels_dir, nome_base + ".txt")

        if not os.path.exists(label):
            erros_validacao.append(
                f"{split}: imagem sem label -> {imagem}"
            )

    # Verificar conteúdo dos labels
    for label_file in labels:

        caminho = os.path.join(labels_dir, label_file)

        with open(caminho, "r") as f:
            linhas = f.readlines()

        if len(linhas) == 0:
            erros_validacao.append(
                f"{split}: label vazio -> {label_file}"
            )
            continue

        for numero, linha in enumerate(linhas, start=1):

            valores = linha.strip().split()

            # YOLO detecção deve ter 5 valores
            if len(valores) != 5:
                erros_validacao.append(
                    f"{split}: {label_file}, linha {numero}: "
                    f"{len(valores)} valores"
                )
                continue

            try:
                classe = int(valores[0])
                coordenadas = [float(v) for v in valores[1:]]

                if classe not in classes_validas:
                    erros_validacao.append(
                        f"{split}: {label_file}, classe inválida: {classe}"
                    )

                # Coordenadas YOLO devem estar entre 0 e 1
                for valor in coordenadas:

                    if valor < 0 or valor > 1:
                        erros_validacao.append(
                            f"{split}: {label_file}, "
                            f"coordenada fora do intervalo: {valor}"
                        )

            except ValueError:
                erros_validacao.append(
                    f"{split}: {label_file}, "
                    f"valor não numérico"
                )

print("\n" + "=" * 50)
print("RESULTADO DA VALIDAÇÃO")
print("=" * 50)

print("Quantidade de erros:", len(erros_validacao))

if erros_validacao:
    print("\nPrimeiros erros encontrados:")
    for erro in erros_validacao[:20]:
        print("-", erro)
else:
    print("Dataset de detecção validado com sucesso! ✅")

**3.3 Criação do data.yaml**

In [ ]:
yaml_content = """train: /content/PPE_dataset_detection/train/images
val: /content/PPE_dataset_detection/valid/images
test: /content/PPE_dataset_detection/test/images

nc: 5

names:
  0: person
  1: with-helmet
  2: with-suit
  3: without-helmet
  4: without-suit
"""

yaml_path = "/content/PPE_dataset_detection/data.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("data.yaml criado com sucesso! ✅")
print("\nConteúdo:")
print(yaml_content)

In [ ]:
#visualizar bounding boxes
import random
import cv2
import matplotlib.pyplot as plt

# Classes
class_names = {
    0: "person",
    1: "with-helmet",
    2: "with-suit",
    3: "without-helmet",
    4: "without-suit"
}

# Pasta de treino
images_dir = os.path.join(
    detection_path,
    "train",
    "images"
)

labels_dir = os.path.join(
    detection_path,
    "train",
    "labels"
)

# Selecionar 4 imagens aleatórias
imagens = [
    f for f in os.listdir(images_dir)
    if f.lower().endswith(image_extensions)
]

selecionadas = random.sample(imagens, 4)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.flatten()

for ax, nome_imagem in zip(axes, selecionadas):

    imagem_path = os.path.join(images_dir, nome_imagem)
    label_path = os.path.join(
        labels_dir,
        os.path.splitext(nome_imagem)[0] + ".txt"
    )

    # Ler imagem
    imagem = cv2.imread(imagem_path)
    imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)

    altura, largura = imagem.shape[:2]

    # Ler labels
    with open(label_path, "r") as f:
        linhas = f.readlines()

    for linha in linhas:

        valores = linha.strip().split()

        classe = int(valores[0])

        x_centro = float(valores[1]) * largura
        y_centro = float(valores[2]) * altura
        box_largura = float(valores[3]) * largura
        box_altura = float(valores[4]) * altura

        x1 = int(x_centro - box_largura / 2)
        y1 = int(y_centro - box_altura / 2)
        x2 = int(x_centro + box_largura / 2)
        y2 = int(y_centro + box_altura / 2)

        cv2.rectangle(
            imagem,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2
        )

        cv2.putText(
            imagem,
            class_names[classe],
            (x1, max(y1 - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 0, 0),
            2
        )

    ax.imshow(imagem)
    ax.set_title(nome_imagem)
    ax.axis("off")

plt.tight_layout()
plt.show()

**3.4 Treinamento**

In [ ]:
#carregando YOLO
from ultralytics import YOLO
import torch

# Carregar modelo pré-treinado
model = YOLO("yolo11n.pt")

print("Modelo carregado com sucesso! ✅")
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
#Treinamento do modelo
results = model.train(
    data="/content/PPE_dataset_detection/data.yaml",

    epochs=30,
    imgsz=640,
    batch=16,

    device=0,
    seed=42,

    project="/content/results/detection",
    name="yolo11n_ppe",

    patience=10,

    plots=True,
    verbose=True
)

**3.5 Avaliação do conjunto de teste**

In [ ]:
from ultralytics import YOLO

# Carregar o melhor modelo encontrado durante o treinamento
best_model = YOLO(
    "/content/results/detection/yolo11n_ppe/weights/best.pt"
)

# Avaliar no conjunto de teste
test_metrics = best_model.val(
    data="/content/PPE_dataset_detection/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/results/detection",
    name="test"
)

print("\nAvaliação no conjunto de TESTE concluída! ✅")

**3.6 Arquivos gerados**

In [ ]:
import os

results_path = "/content/results/detection/yolo11n_ppe"

print("Arquivos gerados pelo treinamento:\n")

for root, dirs, files in os.walk(results_path):
    for file in files:
        print(os.path.join(root, file))

**3.7 Matriz de confusão**

In [ ]:
from IPython.display import display
from PIL import Image

confusion_path = (
    "/content/results/detection/yolo11n_ppe/"
    "confusion_matrix.png"
)

display(Image.open(confusion_path))

In [ ]:
confusion_normalized_path = (
    "/content/results/detection/yolo11n_ppe/"
    "confusion_matrix_normalized.png"
)

display(Image.open(confusion_normalized_path))

**3.8 Previsões**

In [ ]:
from IPython.display import display
from PIL import Image

base_path = "/content/results/detection/yolo11n_ppe"

print("ANOTAÇÕES REAIS")
display(
    Image.open(
        f"{base_path}/val_batch0_labels.jpg"
    )
)

print("PREVISÕES DO MODELO")
display(
    Image.open(
        f"{base_path}/val_batch0_pred.jpg"
    )
)

In [ ]:
#val_batch1
from IPython.display import display
from PIL import Image

base_path = "/content/results/detection/yolo11n_ppe"

print("ANOTAÇÕES REAIS — BATCH 1")
display(
    Image.open(
        f"{base_path}/val_batch1_labels.jpg"
    )
)

print("PREVISÕES DO MODELO — BATCH 1")
display(
    Image.open(
        f"{base_path}/val_batch1_pred.jpg"
    )
)

In [ ]:
#val batch2
print("ANOTAÇÕES REAIS — BATCH 2")
display(
    Image.open(
        f"{base_path}/val_batch2_labels.jpg"
    )
)

print("PREVISÕES DO MODELO — BATCH 2")
display(
    Image.open(
        f"{base_path}/val_batch2_pred.jpg"
    )
)

**3.9 Curvas de treinamento**

In [ ]:
from IPython.display import display
from PIL import Image

results_path = "/content/results/detection/yolo11n_ppe"

display(
    Image.open(
        f"{results_path}/results.png"
    )
)

**3.9.1 Resultados quantitativos**

In [ ]:
import pandas as pd

results_csv = "/content/results/detection/yolo11n_ppe/results.csv"

df_results = pd.read_csv(results_csv)

# Remover espaços dos nomes das colunas
df_results.columns = df_results.columns.str.strip()

print("Colunas disponíveis:")
for coluna in df_results.columns:
    print("-", coluna)

print("\nÚltimas 5 épocas:")
display(df_results.tail())

In [ ]:
#melhores resultados do treinamento
# Melhor época para cada métrica

melhor_precision = df_results.loc[
    df_results["metrics/precision(B)"].idxmax()
]

melhor_recall = df_results.loc[
    df_results["metrics/recall(B)"].idxmax()
]

melhor_map50 = df_results.loc[
    df_results["metrics/mAP50(B)"].idxmax()
]

melhor_map5095 = df_results.loc[
    df_results["metrics/mAP50-95(B)"].idxmax()
]

print("MELHORES RESULTADOS DURANTE O TREINAMENTO")
print("=" * 55)

print(
    f"Melhor Precision: "
    f"época {int(melhor_precision['epoch']) + 1} "
    f"→ {melhor_precision['metrics/precision(B)']:.4f}"
)

print(
    f"Melhor Recall: "
    f"época {int(melhor_recall['epoch']) + 1} "
    f"→ {melhor_recall['metrics/recall(B)']:.4f}"
)

print(
    f"Melhor mAP@0.5: "
    f"época {int(melhor_map50['epoch']) + 1} "
    f"→ {melhor_map50['metrics/mAP50(B)']:.4f}"
)

print(
    f"Melhor mAP@0.5:0.95: "
    f"época {int(melhor_map5095['epoch']) + 1} "
    f"→ {melhor_map5095['metrics/mAP50-95(B)']:.4f}"
)

In [ ]:
#F1-score
# Resultados do conjunto de teste
resultados_teste = {
    "person": (0.963, 0.988),
    "with-helmet": (0.855, 0.924),
    "with-suit": (0.895, 0.970),
    "without-helmet": (1.000, 0.999),
    "without-suit": (0.816, 0.805),
}

print("F1-score por classe")
print("=" * 40)

for classe, (precision, recall) in resultados_teste.items():

    f1 = 2 * (precision * recall) / (precision + recall)

    print(
        f"{classe:18s} "
        f"Precision={precision:.3f} "
        f"Recall={recall:.3f} "
        f"F1={f1:.3f}"
    )

**Resultado oficial do detector**

YOLO11n — PPE
- Precision = 90,6%
- Recall = 93,7%
- mAP@0.5 = 96,0%
- mAP@0.5:0.95 = 78,3%